In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [7]:
matched_path = "../data/raw/judicial_decisions_matched.csv"
lags_path = "../data/processed/judicial_decisions_merged_with_lags.csv"

df_matched = pd.read_csv(matched_path)
df_lags = pd.read_csv(lags_path)

print("Matched shape:", df_matched.shape)
print("Lags shape:", df_lags.shape)

Matched shape: (1077, 214)
Lags shape: (2601, 76)


In [8]:
common_cols = sorted(set(df_matched.columns) & set(df_lags.columns))
matched_only = sorted(set(df_matched.columns) - set(df_lags.columns))
lags_only = sorted(set(df_lags.columns) - set(df_matched.columns))

print("Common columns:", len(common_cols))
print(common_cols)

print("\nMatched-only columns:", len(matched_only))
print(matched_only[:100])

print("\nLags-only columns:", len(lags_only))
print(lags_only[:100])

Common columns: 26
['authorities', 'case_id', 'case_id_words', 'case_name', 'case_number', 'case_significance', 'case_status', 'corresponding_law_reference', 'country', 'decision_cited_in', 'decision_date_raw', 'decision_direction', 'decision_overview', 'facts', 'judicial_body', 'mode_of_expression', 'related_text', 'ruling_judge', 'ruling_judge_evidence', 'ruling_judge_status', 'source_dataset', 'source_url', 'summary_outcome', 'theme', 'type_of_law', 'year']

Matched-only columns: 188
['_merge', 'appeal_nonreg', 'appeal_nonreg_crisis', 'appeal_reg', 'appeal_reg_crisis', 'c_disinfo_gen', 'c_disinfo_gen_crisis', 'c_disinfo_gen_digi', 'c_disinfo_iccpr', 'c_disinfo_iccpr_crisis', 'c_disinfo_iccpr_digi', 'c_disinfo_noniccpr', 'c_disinfo_noniccpr_crisis', 'c_disinfo_noniccpr_digi', 'c_express_iccpr', 'c_express_iccpr_crisis', 'c_express_iccpr_digi', 'c_express_iccpr_govoff', 'c_express_noniccpr', 'c_express_noniccpr_crisis', 'c_express_noniccpr_digi', 'c_malinfo_gen', 'c_malinfo_gen_crisis

In [9]:
key = "case_id"

print("Matched unique:", df_matched[key].nunique(), "/", len(df_matched))
print("Lags unique:", df_lags[key].nunique(), "/", len(df_lags))
print("Matched duplicate rows:", df_matched.duplicated(key).sum())
print("Lags duplicate rows:", df_lags.duplicated(key).sum())

Matched unique: 1077 / 1077
Lags unique: 2601 / 2601
Matched duplicate rows: 0
Lags duplicate rows: 0


In [10]:
matched_ids = set(df_matched[key].dropna())
lags_ids = set(df_lags[key].dropna())

print("IDs in both:", len(matched_ids & lags_ids))
print("Only in matched:", len(matched_ids - lags_ids))
print("Only in lags:", len(lags_ids - matched_ids))

IDs in both: 1077
Only in matched: 0
Only in lags: 1524


In [11]:
df_matched["case_id"] = df_matched["case_id"].astype(str).str.strip()
df_lags["case_id"] = df_lags["case_id"].astype(str).str.strip()

In [12]:
df_merged = df_matched.merge(
    df_lags,
    on="case_id",
    how="left",
    suffixes=("", "_lags"),
    validate="one_to_one"
)

print("Merged shape:", df_merged.shape)

Merged shape: (1077, 289)


In [13]:
shared_cols = [
    'authorities', 'case_id_words', 'case_name', 'case_number',
    'case_significance', 'case_status', 'corresponding_law_reference',
    'country', 'decision_cited_in', 'decision_date_raw',
    'decision_direction', 'decision_overview', 'facts',
    'judicial_body', 'mode_of_expression', 'related_text',
    'ruling_judge', 'ruling_judge_evidence', 'ruling_judge_status',
    'source_dataset', 'source_url', 'summary_outcome',
    'theme', 'type_of_law', 'year'
]

for col in shared_cols:
    lag_col = f"{col}_lags"
    if lag_col in df_merged.columns:
        df_merged[col] = df_merged[col].combine_first(df_merged[lag_col])
        df_merged.drop(columns=[lag_col], inplace=True)

In [14]:
print("Rows:", len(df_merged))
print("Unique case_id:", df_merged["case_id"].nunique())
print("Duplicate case_id:", df_merged.duplicated("case_id").sum())

Rows: 1077
Unique case_id: 1077
Duplicate case_id: 0


In [17]:
output_path = "../data/processed/judicial_decisions_merged_lags_dejure.csv"

df_merged.to_csv(output_path, index=False)